# DSFB Structural Semiotics Calculus (DSSC) — Canonical Figures

**Invariant Forge LLC** | Version 0.1.0 | April 2026

This notebook reproduces the ten canonical figures of the DSFB Structural Semiotics Calculus crate (`dsfb-semiotics-calculus`). All figures are generated from the same deterministic synthetic trajectory and envelope parameters used by the Rust binary.

---

### DSSC framework summary

| Construct | Symbol | Type | Theorem |
|-----------|--------|------|---------|
| Residual sign triple | σ(k) = (‖r(k)‖, ṙ(k), r̈(k)) | `ResidualSign` | §2.1 |
| Admissibility envelope | E ⊆ V | `AdmissibilityEnvelope` | Def 4.1 |
| Grammar FSM | δ: G × Σ → G (total) | `GrammarFsm` | Thm 3.1 |
| Endoductive operator | ℰ: Traj → Ep (total) | `Enduce` trait | Cor 5.4 |
| ProvenanceTag | φ = (σ, g, α, range) | `ProvenanceTag` | Thm 8.3 |
| Observer | 𝒪: Traj → Ep (pure) | `Observer<E>` | Thm 3.2 |
| Grammar Fusion | G₁ ⋈ G₂ | `GrammarFusion` | Prop 7.1 |

### Safety-case properties

| Property | Enforcement |
|----------|-------------|
| SC-1 Determinism | `Enduce::enduce` returns `Episode`, never `Option` |
| SC-2 Non-Interference | `Observer` holds no `&mut` (Rust borrow checker) |
| SC-3 Auditability | Every `Episode` carries full `ProvenanceTag` |
| SC-4 Coverage | `GrammarFsm::step` is total |
| SC-5 No Silent Failure | `Motif::Unknown` with provenance, never `None` |
| SC-6 Graceful Degradation | Impulsive inputs produce `Unknown`, not panics |

> **IP Notice:** Apache 2.0 applies to the software artifact. The underlying DSSC theory is proprietary Background IP of Invariant Forge LLC (DE LLC No. 10529072). Commercial deployment requires a written license. Inquiries: licensing@invariantforge.net

In [ ]:
# Install dependencies (Colab already has matplotlib/numpy; fpdf2 for PDF)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'fpdf2', '--quiet'])
print('Dependencies ready.')

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patches as patches
from matplotlib.patches import FancyArrowPatch, Arc, FancyBboxPatch
from matplotlib.lines import Line2D
from fpdf import FPDF
import zipfile, json, os, pathlib

matplotlib.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'figure.dpi': 150,
})

# ── output directory
OUT = pathlib.Path('dsfb_sc_artifacts')
OUT.mkdir(exist_ok=True)

# ── colour palette (matches Rust crate exactly)
C_ADM  = '#1a7a4a'   # green
C_BDY  = '#c47c00'   # amber
C_VIO  = '#c0392b'   # red
C_MID  = '#2c3e50'   # near-black
C_ACC  = '#2980b9'   # blue
C_GREY = '#555555'

# ── DSSC core logic in Python (mirrors Rust types exactly)

def residual_sign(r, r_prev, r_prev2):
    """σ(k) = (‖r(k)‖, ṙ(k), r̈(k))"""
    magnitude = abs(r)
    drift = r - r_prev
    slew = drift - (r_prev - r_prev2)
    return magnitude, drift, slew

def classify_envelope(magnitude, rho_min, rho_max, delta):
    """EnvelopeRegion: Interior / Boundary / Exterior"""
    if magnitude <= rho_max - delta:
        return 'Admissible'
    elif magnitude <= rho_max + delta:
        return 'Boundary'
    else:
        return 'Violation'

def compute_grammar_trace(residuals, rho_min=0.1, rho_max=1.0, delta=0.02):
    """Run GrammarFsm over residual sequence; return (signs, states, persistences)"""
    signs, states, persistences = [], [], []
    prev = prev2 = 0.0
    current_state = 'Admissible'
    pers = 0
    for r in residuals:
        mag, drift, slew = residual_sign(r, prev, prev2)
        signs.append((mag, drift, slew))
        new_state = classify_envelope(mag, rho_min, rho_max, delta)
        if new_state == current_state:
            pers += 1
        else:
            current_state = new_state
            pers = 1
        states.append(current_state)
        persistences.append(pers)
        prev2, prev = prev, r
    return signs, states, persistences

STATE_COLOR = {'Admissible': C_ADM, 'Boundary': C_BDY, 'Violation': C_VIO}

# ── Canonical synthetic trajectory (identical to Rust crate)
TRAJ = np.array([
    # Nominal phase
    0.15, 0.22, 0.18, 0.28, 0.20, 0.25, 0.19, 0.30, 0.22, 0.26, 0.21, 0.24,
    # Boundary approach
    0.55, 0.68, 0.78, 0.85, 0.92,
    # Violation burst 1
    1.05, 1.18, 1.10, 1.22,
    # Recovery to nominal
    0.80, 0.60, 0.40, 0.28, 0.22, 0.19, 0.25, 0.20, 0.27, 0.21, 0.23, 0.20, 0.22,
    # Second violation burst
    0.70, 0.88, 1.04, 1.15, 1.08,
    # Final recovery
    0.72, 0.50, 0.30, 0.22, 0.18,
])

SIGNS, STATES, PERSISTENCES = compute_grammar_trace(TRAJ)
N = len(TRAJ)
K = np.arange(N)

print(f'Trajectory: {N} steps')
from collections import Counter
dist = Counter(STATES)
for s, c in sorted(dist.items()):
    print(f'  {s}: {c} steps ({100*c/N:.1f}%)')

In [ ]:
# ════════════════════════════════════════════════════════
# Figure 1 — Residual Sign Triple
# ════════════════════════════════════════════════════════
mags   = np.array([s[0] for s in SIGNS])
drifts = np.array([s[1] for s in SIGNS])
slews  = np.array([s[2] for s in SIGNS])

fig, axes = plt.subplots(3, 1, figsize=(11, 6.5), sharex=True)
fig.suptitle('Figure 1 — Residual Sign Triple: σ(k) = (‖r(k)‖, ṙ(k), r̈(k))',
             fontsize=12, fontweight='bold', y=1.01)

for ax, vals, color, ylabel in zip(
    axes,
    [mags, drifts, slews],
    [C_ACC, C_VIO, C_BDY],
    ['‖r(k)‖  (magnitude)', 'ṙ(k)  (drift)', 'r̈(k)  (slew)']
):
    ax.plot(K, vals, color=color, lw=1.8, marker='o', ms=3, markerfacecolor=color)
    ax.axhline(0, color='#aaa', lw=0.7, ls='--')
    ax.set_ylabel(ylabel, fontsize=10)
    ax.tick_params(labelsize=9)

axes[-1].set_xlabel('step k', fontsize=10)
fig.text(0.5, -0.04,
    'Magnitude (top), drift ṙ (middle), slew r̈ (bottom) across synthetic trajectory',
    ha='center', fontsize=8, color=C_GREY)
plt.tight_layout()
plt.savefig(OUT / 'fig01_residual_sign_triple.svg', bbox_inches='tight')
plt.savefig(OUT / 'fig01_residual_sign_triple.png', bbox_inches='tight')
plt.show()
print('Figure 1 saved.')

In [ ]:
# ════════════════════════════════════════════════════════
# Figure 2 — Admissibility Envelope Geometry
# ════════════════════════════════════════════════════════
rho_min, rho_max, delta = 0.25, 1.0, 0.04

fig, ax = plt.subplots(figsize=(6, 6))
ax.set_aspect('equal')
ax.set_xlim(-1.55, 1.55); ax.set_ylim(-1.55, 1.55)
ax.set_title(
    'Figure 2 — Admissibility Envelope Geometry\n'
    'B(0,ρ_min) ⊆ E = B(0,ρ_max); δ-band boundary layer',
    fontsize=10, fontweight='bold')

# Exterior δ-band
outer_outer = plt.Circle((0, 0), rho_max + delta, color=C_VIO, alpha=0.1, lw=0)
ax.add_patch(outer_outer)

# Interior
inner = plt.Circle((0, 0), rho_max - delta, color=C_BDY, alpha=0.15, lw=0)
ax.add_patch(inner)

# Admissible core
core = plt.Circle((0, 0), rho_min, color=C_ADM, alpha=0.3, lw=0)
ax.add_patch(core)

# Boundary circles
for r, c, ls, lbl in [
    (rho_max + delta, C_VIO, '--', f'ρ_max+δ={rho_max+delta:.2f}'),
    (rho_max,         C_MID, '-',  f'ρ_max={rho_max:.1f}'),
    (rho_max - delta, C_BDY, '--', f'ρ_max−δ={rho_max-delta:.2f}'),
    (rho_min,         C_ADM, '-',  f'ρ_min={rho_min:.2f}'),
]:
    circle = plt.Circle((0, 0), r, fill=False, ec=c, lw=1.8, ls=ls, label=lbl)
    ax.add_patch(circle)

# Labels
ax.text(0, 0, '0', ha='center', va='center', fontsize=11, color=C_MID, fontweight='bold')
ax.text(0, rho_min*0.5, 'Interior\n(Adm)', ha='center', va='center', fontsize=10,
        color=C_ADM, fontweight='bold')
ax.text(0.78, 0.78, 'δ-band\n(Bdy)', ha='center', fontsize=9, color=C_BDY)
ax.text(0, 1.35, 'Exterior\n(Vio)', ha='center', fontsize=9, color=C_VIO)

ax.set_axis_off()
ax.legend(loc='lower right', fontsize=8)
fig.text(0.5, 0.01, 'δ ≤ ρ_min/4 (Definition 4.1); region → grammar state mapping',
         ha='center', fontsize=8, color=C_GREY)
plt.tight_layout()
plt.savefig(OUT / 'fig02_admissibility_envelope.svg', bbox_inches='tight')
plt.savefig(OUT / 'fig02_admissibility_envelope.png', bbox_inches='tight')
plt.show()
print('Figure 2 saved.')

In [ ]:
# ════════════════════════════════════════════════════════
# Figure 3 — Grammar FSM Diagram
# ════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.set_xlim(0, 9); ax.set_ylim(0, 4)
ax.set_title('Figure 3 — Grammar FSM: G = {Adm, Bdy, Vio}, δ: G × Σ → G (total)',
             fontsize=10, fontweight='bold')
ax.set_axis_off()

NODES = {'Adm': (1.8, 2.0), 'Bdy': (4.5, 2.0), 'Vio': (7.2, 2.0)}
COLORS = {'Adm': C_ADM, 'Bdy': C_BDY, 'Vio': C_VIO}
SUBLBL = {'Adm': 'Nominal', 'Bdy': 'Early-warning', 'Vio': 'ℰ fires'}

for name, (x, y) in NODES.items():
    c = COLORS[name]
    circle = plt.Circle((x, y), 0.75, color=c, alpha=0.18, zorder=2)
    ax.add_patch(circle)
    circle2 = plt.Circle((x, y), 0.75, fill=False, ec=c, lw=2.5, zorder=3)
    ax.add_patch(circle2)
    ax.text(x, y + 0.15, name, ha='center', va='center', fontsize=13,
            fontweight='bold', color=c, zorder=4)
    ax.text(x, y - 0.22, SUBLBL[name], ha='center', va='center', fontsize=8,
            color=c, zorder=4)

def curved_arrow(ax, x1, y1, x2, y2, color, label, bend=0.3, above=True):
    mid_x = (x1 + x2) / 2
    mid_y = (y1 + y2) / 2 + (bend if above else -bend)
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.8,
                                connectionstyle=f'arc3,rad={0.3 if above else -0.3}'))
    ax.text(mid_x, mid_y + (0.1 if above else -0.1), label,
            ha='center', fontsize=8.5, color=color)

# Forward: Adm→Bdy, Bdy→Vio
curved_arrow(ax, 2.55, 2.3, 3.75, 2.3, C_BDY, 'Boundary', bend=0.5)
curved_arrow(ax, 5.25, 2.3, 6.45, 2.3, C_VIO, 'Exterior', bend=0.5)
# Backward: Bdy→Adm
curved_arrow(ax, 3.75, 1.7, 2.55, 1.7, C_ADM, 'Interior', bend=-0.5, above=False)
# Reset: Vio→Adm
ax.annotate('', xy=(1.8, 1.25), xytext=(7.2, 1.25),
            arrowprops=dict(arrowstyle='->', color='#888', lw=1.2, ls='dashed',
                            connectionstyle='arc3,rad=0.0'))
ax.text(4.5, 0.85, 'reset (ℰ emitted)', ha='center', fontsize=8, color='#888')
# Self-loop on Adm
self_loop = Arc((1.8, 2.0), 1.6, 1.2, angle=0, theta1=130, theta2=220, color=C_ADM, lw=1.2, ls='dashed')
ax.add_patch(self_loop)
ax.text(0.5, 2.9, 'Interior', fontsize=8, color=C_ADM)

ax.text(4.5, 0.15, 'Theorem 3.1 (Totality): δ is defined for every (g, σ) pair — no undefined transitions.',
        ha='center', fontsize=8, color=C_GREY)
plt.tight_layout()
plt.savefig(OUT / 'fig03_grammar_fsm_diagram.svg', bbox_inches='tight')
plt.savefig(OUT / 'fig03_grammar_fsm_diagram.png', bbox_inches='tight')
plt.show()
print('Figure 3 saved.')

In [ ]:
# ════════════════════════════════════════════════════════
# Figure 4 — Grammar State Trajectory
# ════════════════════════════════════════════════════════
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 5.5),
                                 gridspec_kw={'height_ratios': [4, 1]}, sharex=True)
fig.suptitle('Figure 4 — Grammar State Sequence Overlaid on Residual Trajectory',
             fontsize=11, fontweight='bold')

# Residual trace — segment colour by state
for i in range(1, N):
    col = STATE_COLOR[STATES[i]]
    ax1.plot(K[i-1:i+1], TRAJ[i-1:i+1], color=col, lw=2.2)
    ax1.scatter([K[i]], [TRAJ[i]], color=col, s=18, zorder=3)

ax1.axhline(1.0, color=C_MID, lw=1.0, ls='--', label='ρ_max = 1.0')
ax1.axhspan(0.98, 1.02, alpha=0.08, color=C_BDY, label='δ-band')
ax1.set_ylabel('‖r(k)‖', fontsize=10)
ax1.legend(fontsize=8, loc='upper left')
ax1.set_ylim(0, 1.4)

# Grammar state band
state_num = {'Admissible': 0, 'Boundary': 1, 'Violation': 2}
for i in range(N):
    ax2.bar(K[i], 1, color=STATE_COLOR[STATES[i]], alpha=0.7, width=1.0, align='edge')
ax2.set_yticks([])
ax2.set_xlabel('step k', fontsize=10)
ax2.set_yticklabels([])

legend_els = [mpatches.Patch(color=C_ADM, label='Adm'),
              mpatches.Patch(color=C_BDY, label='Bdy'),
              mpatches.Patch(color=C_VIO, label='Vio')]
ax2.legend(handles=legend_els, loc='upper right', fontsize=8, ncol=3)

plt.tight_layout()
plt.savefig(OUT / 'fig04_grammar_state_trajectory.svg', bbox_inches='tight')
plt.savefig(OUT / 'fig04_grammar_state_trajectory.png', bbox_inches='tight')
plt.show()
print('Figure 4 saved.')

In [ ]:
# ════════════════════════════════════════════════════════
# Figure 5 — Persistence Counter K(k)
# ════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(11, 4))
ax.set_title(
    'Figure 5 — Persistence Counter K(k): Consecutive-Step Dwell in Grammar State',
    fontsize=10, fontweight='bold')

bars = ax.bar(K, PERSISTENCES,
              color=[STATE_COLOR[s] for s in STATES],
              alpha=0.75, edgecolor='none', width=0.9)

ax.set_xlabel('step k', fontsize=10)
ax.set_ylabel('K(k)  (persistence)', fontsize=10)

legend_els = [mpatches.Patch(color=C_ADM, alpha=0.75, label='Admissible'),
              mpatches.Patch(color=C_BDY, alpha=0.75, label='Boundary'),
              mpatches.Patch(color=C_VIO, alpha=0.75, label='Violation')]
ax.legend(handles=legend_els, fontsize=9)

fig.text(0.5, -0.04, 'K(k) = consecutive steps in current grammar state; resets on transition. '
         'Persistence used by HeuristicsBank pattern matching (min_persistence criterion).',
         ha='center', fontsize=8, color=C_GREY)
plt.tight_layout()
plt.savefig(OUT / 'fig05_persistence_counter.svg', bbox_inches='tight')
plt.savefig(OUT / 'fig05_persistence_counter.png', bbox_inches='tight')
plt.show()
print('Figure 5 saved.')

In [ ]:
# ════════════════════════════════════════════════════════
# Figure 6 — Endoductive Operator Data-Flow
# ════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(9, 5))
ax.set_xlim(0, 10); ax.set_ylim(0, 6)
ax.set_title('Figure 6 — Endoductive Operator ℰ: Full Data-Flow Diagram',
             fontsize=10, fontweight='bold')
ax.set_axis_off()

def box(ax, x, y, w, h, label, sublabel, facecolor, edgecolor):
    r = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.05',
                        facecolor=facecolor, edgecolor=edgecolor, lw=1.8, zorder=2)
    ax.add_patch(r)
    ax.text(x + w/2, y + h/2 + 0.1, label, ha='center', va='center',
            fontsize=10, fontweight='bold', color=C_MID, zorder=3)
    if sublabel:
        ax.text(x + w/2, y + h/2 - 0.28, sublabel, ha='center', va='center',
                fontsize=7.5, color='#555', zorder=3)

INPUTS = [
    ('Trajectory r', '‖r(k₀:k*)‖', 1.2),
    ('Sign Sequence σ', '(‖r‖, ṙ, r̈) per step', 2.4),
    ('Grammar Path g', 'Adm/Bdy/Vio per step', 3.6),
    ('Heuristics Bank h', 'Monotone pattern store', 4.8),
]
for lbl, sub, y in INPUTS:
    box(ax, 0.2, y - 0.4, 2.5, 0.75, lbl, sub,
        f'#e8f4fb', C_ACC)
    ax.annotate('', xy=(4.2, 3.5), xytext=(2.7, y - 0.02),
                arrowprops=dict(arrowstyle='->', color=C_ACC, lw=1.1, ls='dashed',
                                connectionstyle='arc3,rad=0.0'))

# Central ℰ box
box(ax, 4.2, 2.9, 1.6, 1.2, 'ℰ', 'endoductive op.', '#f0f0f0', C_MID)

# Outputs
box(ax, 6.5, 4.0, 2.5, 0.75, 'Motif m', 'Named or Unknown', '#e8f8f0', C_ADM)
box(ax, 6.5, 2.6, 2.5, 0.75, 'ProvenanceTag φ', '(σ, g, α, range)', '#e8f4fb', C_ACC)
ax.annotate('', xy=(6.5, 4.38), xytext=(5.8, 3.9),
            arrowprops=dict(arrowstyle='->', color=C_MID, lw=1.5))
ax.annotate('', xy=(6.5, 2.98), xytext=(5.8, 3.2),
            arrowprops=dict(arrowstyle='->', color=C_MID, lw=1.5))

# Episode bracket
epbr = FancyBboxPatch((6.3, 2.3), 3.0, 2.75, boxstyle='round,pad=0.08',
                       facecolor='none', edgecolor=C_MID, lw=1.5, ls='dashed', zorder=1)
ax.add_patch(epbr)
ax.text(7.8, 5.1, 'Episode (m, φ)', ha='center', fontsize=9, color=C_MID)

# Guarantee banner
box(ax, 6.3, 1.4, 3.0, 0.65, 'Corollary 5.4', '∀ input → Episode (total; no panic; no None)',
    '#fffde7', C_BDY)

ax.text(5.0, 0.3, 'Type-level totality: Enduce::enduce returns Episode — not Option<Episode>, not Result.',
        ha='center', fontsize=7.5, color=C_GREY)
plt.tight_layout()
plt.savefig(OUT / 'fig06_endoductive_operator.svg', bbox_inches='tight')
plt.savefig(OUT / 'fig06_endoductive_operator.png', bbox_inches='tight')
plt.show()
print('Figure 6 saved.')

In [ ]:
# ════════════════════════════════════════════════════════
# Figure 7 — ProvenanceTag Anatomy
# ════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.set_xlim(0, 10); ax.set_ylim(0, 7)
ax.set_title('Figure 7 — ProvenanceTag φ: Deterministic Audit Certificate (Theorem 8.3)',
             fontsize=10, fontweight='bold')
ax.set_axis_off()

# Outer episode frame
outer = FancyBboxPatch((0.3, 0.5), 9.4, 6.0, boxstyle='round,pad=0.1',
                        facecolor='none', edgecolor=C_MID, lw=2.0)
ax.add_patch(outer)
ax.text(5.0, 6.7, 'Episode (m, φ)', ha='center', fontsize=11, fontweight='bold', color=C_MID)

# Motif panel
motif_bg = FancyBboxPatch((0.5, 0.7), 2.2, 5.6, boxstyle='round,pad=0.05',
                           facecolor='#e8f8f0', edgecolor=C_ADM, lw=1.5)
ax.add_patch(motif_bg)
ax.text(1.6, 6.0, 'Motif m', ha='center', fontsize=10, fontweight='bold', color=C_ADM)
for y, txt in [(5.3, 'Named("drift_up")'), (4.9, '— or —'), (4.5, 'Unknown')]:
    ax.text(1.6, y, txt, ha='center', fontsize=9.5, color=C_MID)
ax.text(1.6, 3.8, '(Epistemically honest:', ha='center', fontsize=8, color=C_GREY)
ax.text(1.6, 3.4, 'provenance always', ha='center', fontsize=8, color=C_GREY)
ax.text(1.6, 3.0, 'complete even when', ha='center', fontsize=8, color=C_GREY)
ax.text(1.6, 2.6, 'motif is Unknown)', ha='center', fontsize=8, color=C_GREY)
ax.text(1.6, 1.9, 'Corollary 5.4:', ha='center', fontsize=8, fontweight='bold', color=C_ADM)
ax.text(1.6, 1.5, 'No Silent Failure', ha='center', fontsize=8, color=C_ADM)

# ProvenanceTag fields
prov_bg = FancyBboxPatch((3.0, 0.7), 6.5, 5.6, boxstyle='round,pad=0.05',
                          facecolor='#e8f4fb', edgecolor=C_ACC, lw=1.5)
ax.add_patch(prov_bg)
ax.text(6.25, 6.0, 'ProvenanceTag φ', ha='center', fontsize=10, fontweight='bold', color=C_ACC)

FIELDS = [
    ('sign_sequence',  'σ(k₀),…,σ(k*)',  'Vec<ResidualSign> — observed triple per step'),
    ('grammar_path',   'g(k₀),…,g(k*)',  'Vec<GrammarState> — Adm/Bdy/Vio per step'),
    ('add_descriptor', 'α',               'String — ADD algebraic invariant descriptor'),
    ('step_range',     '(k₀, k*)',        'Episode window indices in trajectory'),
]
for i, (f, notation, desc) in enumerate(FIELDS):
    fy = 5.1 - i * 1.1
    row_bg = FancyBboxPatch((3.15, fy - 0.35), 6.2, 0.85, boxstyle='round,pad=0.03',
                             facecolor='white', edgecolor='#ddd', lw=1.0)
    ax.add_patch(row_bg)
    ax.text(3.3, fy + 0.2, f, fontsize=9.5, fontweight='bold', va='top', color=C_MID)
    ax.text(3.3, fy - 0.15, f'  {notation} — {desc}', fontsize=7.5, va='top', color=C_GREY)

# Replay guarantee banner
replay_bg = FancyBboxPatch((0.3, 0.1), 9.4, 0.7, boxstyle='round,pad=0.05',
                            facecolor='#e8f8f0', edgecolor=C_ADM, lw=1.2)
ax.add_patch(replay_bg)
ax.text(5.0, 0.6, 'Theorem 8.3 — Deterministic Auditability:',
        ha='center', fontsize=8.5, fontweight='bold', color=C_ADM)
ax.text(5.0, 0.25,
        'Any observer re-running ℰ with (φ, h) reproduces the identical Episode exactly.',
        ha='center', fontsize=8, color=C_MID)

plt.tight_layout()
plt.savefig(OUT / 'fig07_provenance_tag_anatomy.svg', bbox_inches='tight')
plt.savefig(OUT / 'fig07_provenance_tag_anatomy.png', bbox_inches='tight')
plt.show()
print('Figure 7 saved.')

In [ ]:
# ════════════════════════════════════════════════════════
# Figure 8 — Bank Monotonicity (Theorem 7.2)
# ════════════════════════════════════════════════════════
steps = np.arange(14)
motifs   = np.where(steps < 2, 0, steps - 1)
patterns = np.where(steps < 2, 0, (steps - 1) * 2)

fig, ax = plt.subplots(figsize=(9, 4))
ax.set_title('Figure 8 — Bank Monotonicity (Theorem 7.2)\n'
             'augment(h, m, P) — never removes', fontsize=10, fontweight='bold')

ax.bar(steps, patterns, color=C_ACC, alpha=0.6, label='Patterns in bank', zorder=2, width=0.8)
ax.plot(steps, motifs, color=C_ADM, lw=2.0, marker='o', ms=5, label='Named motifs', zorder=3)

ax.axvline(1.5, color='#888', lw=1.0, ls='--')
ax.text(1.6, max(patterns) * 0.75,
        'Day-One\nh = ∅, total\n(Prop. 9.1)',
        fontsize=8, color='#888', va='top')

ax.set_xlabel('augmentation step', fontsize=10)
ax.set_ylabel('count', fontsize=10)
ax.legend(fontsize=9)
fig.text(0.5, -0.04,
         'HeuristicsBank.augment() — monotone only: no remove() method. Day-One: h = ∅ is valid (Proposition 9.1).',
         ha='center', fontsize=8, color=C_GREY)
plt.tight_layout()
plt.savefig(OUT / 'fig08_bank_monotonicity.svg', bbox_inches='tight')
plt.savefig(OUT / 'fig08_bank_monotonicity.png', bbox_inches='tight')
plt.show()
print('Figure 8 saved.')

In [ ]:
# ════════════════════════════════════════════════════════
# Figure 9 — Observer Non-Interference
# ════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.set_xlim(0, 10); ax.set_ylim(0, 7)
ax.set_title('Figure 9 — Non-Interference (SC-2): Observer is a Pure Read-Only Functor',
             fontsize=10, fontweight='bold')
ax.set_axis_off()

# Host system
host_bg = FancyBboxPatch((0.3, 1.0), 3.5, 5.5, boxstyle='round,pad=0.1',
                          facecolor='#fff5f5', edgecolor=C_VIO, lw=2.5)
ax.add_patch(host_bg)
ax.text(2.05, 6.2, 'Host System', ha='center', fontsize=11, fontweight='bold', color=C_VIO)
ax.text(2.05, 5.8, '(state S_t — mutable)', ha='center', fontsize=8.5, color=C_GREY)

HOST_ITEMS = ['&mut estimator', '&mut buffer', '&mut history', 'write path active']
for i, item in enumerate(HOST_ITEMS):
    row = FancyBboxPatch((0.5, 4.4 - i * 0.85), 3.1, 0.65,
                          boxstyle='round,pad=0.04', facecolor='white', edgecolor=C_VIO, lw=1.0)
    ax.add_patch(row)
    ax.text(0.7, 4.73 - i * 0.85, item, fontsize=9, color=C_VIO, va='center')

# Read arrow
ax.annotate('', xy=(5.7, 4.0), xytext=(3.8, 4.0),
            arrowprops=dict(arrowstyle='->', color=C_ADM, lw=2.2))
ax.text(4.75, 4.3, '‖r(k)‖', ha='center', fontsize=9.5, color=C_ADM, fontweight='bold')
ax.text(4.75, 3.7, 'shared ref. only', ha='center', fontsize=8, color=C_ADM)

# Observer
obs_bg = FancyBboxPatch((5.7, 1.0), 4.0, 5.5, boxstyle='round,pad=0.1',
                         facecolor='#e8f8f0', edgecolor=C_ADM, lw=2.5)
ax.add_patch(obs_bg)
ax.text(7.7, 6.2, 'Observer 𝒪', ha='center', fontsize=11, fontweight='bold', color=C_ADM)
ax.text(7.7, 5.8, '(pure, no &mut to host)', ha='center', fontsize=8.5, color=C_GREY)

OBS_ITEMS = ['envelope: AdmissibilityEnvelope', 'bank: HeuristicsBank',
             'fsm: GrammarFsm (own)', 'enduce: impl Enduce']
for i, item in enumerate(OBS_ITEMS):
    row = FancyBboxPatch((5.9, 4.4 - i * 0.85), 3.6, 0.65,
                          boxstyle='round,pad=0.04', facecolor='white', edgecolor=C_ADM, lw=1.0)
    ax.add_patch(row)
    ax.text(6.1, 4.73 - i * 0.85, item, fontsize=8.5, color=C_ADM, va='center')

# No write-back (crossed arrow)
ax.annotate('', xy=(3.8, 2.5), xytext=(5.7, 2.5),
            arrowprops=dict(arrowstyle='->', color='#ccc', lw=1.3, ls='dashed'))
ax.plot([4.5, 5.0], [2.2, 2.8], color=C_VIO, lw=2.5)
ax.plot([4.5, 5.0], [2.8, 2.2], color=C_VIO, lw=2.5)
ax.text(4.75, 1.9, 'no write-back\n(structurally impossible)', ha='center', fontsize=8, color=C_VIO)

# Output → Episode
ax.annotate('', xy=(10.2, 4.0), xytext=(9.7, 4.0),
            arrowprops=dict(arrowstyle='->', color=C_MID, lw=2.0))
ax.text(10.3, 4.1, 'Episode\n(m, φ)', fontsize=9, color=C_MID, va='center')

# Guarantee banner
g_bg = FancyBboxPatch((0.3, 0.1), 9.4, 0.75, boxstyle='round,pad=0.05',
                        facecolor='#e8f8f0', edgecolor=C_ADM, lw=1.2)
ax.add_patch(g_bg)
ax.text(5.0, 0.6,
        'Rust ownership enforcement: Observer takes f64 by Copy — no &mut path to host. Borrow checker verifies.',
        ha='center', fontsize=7.5, color=C_MID)

plt.tight_layout()
plt.savefig(OUT / 'fig09_observer_noninterference.svg', bbox_inches='tight')
plt.savefig(OUT / 'fig09_observer_noninterference.png', bbox_inches='tight')
plt.show()
print('Figure 9 saved.')

In [ ]:
# ════════════════════════════════════════════════════════
# Figure 10 — Cross-Stream Grammar Fusion G₁ ⋈ G₂
# ════════════════════════════════════════════════════════
traj2 = np.clip(TRAJ * 0.8 + 0.05 + np.arange(N) % 5 * 0.04, 0, 1.35)
_, states2, _ = compute_grammar_trace(traj2, rho_max=0.88)

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(11, 7),
                                      gridspec_kw={'height_ratios': [3, 3, 1]}, sharex=True)
fig.suptitle('Figure 10 — Cross-Stream Grammar Fusion G₁ ⋈ G₂ (Definition 7.1)',
             fontsize=11, fontweight='bold')

for ax, traj, states, label, rho_m in [
    (ax1, TRAJ, STATES, 'Stream 1  (ρ_max = 1.0)', 1.0),
    (ax2, traj2, states2, 'Stream 2  (ρ_max = 0.88)', 0.88),
]:
    for i in range(1, N):
        ax.plot(K[i-1:i+1], traj[i-1:i+1], color=STATE_COLOR[states[i]], lw=2.0)
        ax.scatter([K[i]], [traj[i]], color=STATE_COLOR[states[i]], s=14, zorder=3)
    ax.axhline(rho_m, color=C_MID, lw=0.8, ls='--', alpha=0.7)
    ax.set_ylabel('‖r(k)‖', fontsize=9)
    ax.set_ylim(0, 1.4)
    ax.text(0.01, 0.92, label, transform=ax.transAxes, fontsize=9, color=C_MID)

# Joint violation band
for i in range(N):
    if STATES[i] == 'Violation' and states2[i] == 'Violation':
        ax3.bar(K[i], 1, color=C_VIO, alpha=0.9, width=1.0, align='edge')
    elif STATES[i] == 'Violation' or states2[i] == 'Violation':
        ax3.bar(K[i], 1, color=C_BDY, alpha=0.5, width=1.0, align='edge')
    else:
        ax3.bar(K[i], 1, color=C_ADM, alpha=0.2, width=1.0, align='edge')

ax3.set_yticks([])
ax3.set_xlabel('step k', fontsize=10)
legend_els = [mpatches.Patch(color=C_VIO, alpha=0.9, label='⋈ joint Vio'),
              mpatches.Patch(color=C_BDY, alpha=0.5, label='single-stream Vio'),
              mpatches.Patch(color=C_ADM, alpha=0.4, label='nominal')]
ax3.legend(handles=legend_els, loc='upper right', fontsize=7.5, ncol=3)

plt.tight_layout()
plt.savefig(OUT / 'fig10_cross_stream_fusion.svg', bbox_inches='tight')
plt.savefig(OUT / 'fig10_cross_stream_fusion.png', bbox_inches='tight')
plt.show()
print('Figure 10 saved.')

In [ ]:
# ════════════════════════════════════════════════════════
# JSON summary
# ════════════════════════════════════════════════════════
from collections import Counter

dist = Counter(STATES)
mags = [s[0] for s in SIGNS]
drifts = [abs(s[1]) for s in SIGNS]

summary = {
    'crate': 'dsfb-semiotics-calculus',
    'version': '0.1.0',
    'notebook': 'dsfb_semiotics_calculus_figures.ipynb',
    'trajectory_length': int(N),
    'envelope': {'rho_min': 0.1, 'rho_max': 1.0, 'delta': 0.02},
    'grammar_state_distribution': dict(dist),
    'sign_statistics': {
        'max_magnitude': float(max(mags)),
        'max_abs_drift': float(max(drifts)),
    },
    'figures_generated': [f'fig{i:02d}_*.svg+png' for i in range(1, 11)],
    'safety_case_properties': [
        'SC-1 Determinism: Enduce::enduce returns Episode (not Option)',
        'SC-2 Non-Interference: Observer holds no &mut; Rust borrow checker',
        'SC-3 Auditability: every Episode carries ProvenanceTag',
        'SC-4 Coverage: GrammarFsm::step is total',
        'SC-5 No Silent Failure: Motif::Unknown with provenance, never None',
        'SC-6 Graceful Degradation: impulsive inputs produce Unknown, not panics',
    ]
}

summary_path = OUT / 'summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print('Summary JSON written.')
print(json.dumps(summary, indent=2))

In [ ]:
# ════════════════════════════════════════════════════════
# PDF artifact report
# ════════════════════════════════════════════════════════
pdf = FPDF()
pdf.set_auto_page_break(auto=True, margin=15)
pdf.add_page()
pdf.set_font('Helvetica', 'B', 16)
pdf.cell(0, 10, 'DSFB Structural Semiotics Calculus', new_x="LMARGIN", new_y="NEXT", align='C')
pdf.set_font('Helvetica', '', 12)
pdf.cell(0, 8, 'Canonical Figures and Artifact Report', new_x="LMARGIN", new_y="NEXT", align='C')
pdf.cell(0, 8, 'Invariant Forge LLC | Version 0.1.0 | April 2026', new_x="LMARGIN", new_y="NEXT", align='C')
pdf.ln(6)

pdf.set_font('Helvetica', 'B', 12)
pdf.cell(0, 8, 'Safety-Case Properties', new_x="LMARGIN", new_y="NEXT")
pdf.set_font('Helvetica', '', 9)
for prop in summary['safety_case_properties']:
    pdf.cell(0, 6, f'  * {prop}', new_x="LMARGIN", new_y="NEXT")
pdf.ln(4)

pdf.set_font('Helvetica', 'B', 12)
pdf.cell(0, 8, 'Trajectory Statistics', new_x="LMARGIN", new_y="NEXT")
pdf.set_font('Helvetica', '', 9)
pdf.cell(0, 6, f'  Trajectory length: {N} steps', new_x="LMARGIN", new_y="NEXT")
for state, count in dist.items():
    pdf.cell(0, 6, f'  {state}: {count} steps ({100*count/N:.1f}%)', new_x="LMARGIN", new_y="NEXT")
pdf.ln(4)

# Embed PNG figures
pdf.set_font('Helvetica', 'B', 12)
pdf.cell(0, 8, 'Canonical Figures', new_x="LMARGIN", new_y="NEXT")
FIG_TITLES = [
    'Figure 1 - Residual Sign Triple',
    'Figure 2 - Admissibility Envelope Geometry',
    'Figure 3 - Grammar FSM Diagram',
    'Figure 4 - Grammar State Trajectory',
    'Figure 5 - Persistence Counter',
    'Figure 6 - Endoductive Operator Data-Flow',
    'Figure 7 - ProvenanceTag Anatomy',
    'Figure 8 - Bank Monotonicity',
    'Figure 9 - Observer Non-Interference',
    'Figure 10 - Cross-Stream Grammar Fusion',
]

for i in range(1, 11):
    png = list(OUT.glob(f'fig{i:02d}_*.png'))
    if not png:
        continue
    pdf.add_page()
    pdf.set_font('Helvetica', 'B', 10)
    pdf.cell(0, 8, FIG_TITLES[i - 1], new_x="LMARGIN", new_y="NEXT")
    pdf.image(str(png[0]), x=10, y=None, w=190)

pdf.add_page()
pdf.set_font('Helvetica', 'B', 11)
pdf.cell(0, 8, 'IP Notice', new_x="LMARGIN", new_y="NEXT")
pdf.set_font('Helvetica', '', 9)
pdf.multi_cell(0, 6,
    'Apache 2.0 applies to this software artifact as an executable and distributable work. '
    'It does not constitute a license to the underlying theoretical framework, mathematical '
    'architecture, formal constructions, or supervisory methods described in the companion paper, '
    'which constitute proprietary Background IP of Invariant Forge LLC (Delaware LLC No. 10529072). '
    'Commercial deployment requires a separate written license. '
    'Inquiries: licensing@invariantforge.net')

pdf_path = OUT / 'dsfb-semiotics-calculus-report.pdf'
pdf.output(str(pdf_path))
print(f'PDF written: {pdf_path}')

In [ ]:
# ════════════════════════════════════════════════════════
# ZIP bundle
# ════════════════════════════════════════════════════════
zip_path = OUT / 'dsfb-semiotics-calculus-artifacts.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(OUT.iterdir()):
        if f.suffix in {'.svg', '.png', '.pdf', '.json', '.md'}:
            zf.write(f, arcname=f.name)

file_list = []
with zipfile.ZipFile(zip_path) as zf:
    for info in zf.infolist():
        file_list.append(f'  {info.filename}  ({info.file_size:,} B)')

print(f'ZIP bundle: {zip_path}')
print(f'Contents ({len(file_list)} files):')
for l in file_list:
    print(l)

In [ ]:
# ════════════════════════════════════════════════════════
# Download prompt (Colab)
# ════════════════════════════════════════════════════════
try:
    from google.colab import files
    files.download(str(zip_path))
    print('Download initiated.')
except ImportError:
    print(f'Not in Colab. ZIP available at: {zip_path.resolve()}')